# 04 — Collaborative Filtering Experiments

Owner: **18- Thanh Loan** (Pipeline) — Tasks T06–T10 / T15–T19.

Goals:
- Inspect sparse utility matrix (shape, density)
- Spot-check item-item similarity
- Compare HR@10 / NDCG@10 across sample sizes

Run AFTER `scripts/build_cf_artifacts.py` has produced artifacts.

In [ ]:
from pathlib import Path
import sys
import time

import pandas as pd
from scipy.sparse import issparse

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from data_processing import load_processed
from recommender_cf import load_cf_artifacts, recommend_for_user, build_utility_matrix
from evaluation import run_evaluation

print("Setup OK")
print("artifacts:", (ROOT / "artifacts" / "item_similarity.npz").exists())

## T15 — Utility matrix density

Load CF ratings and report shape / nnz / density.

In [ ]:
movies, ratings_cf, ratings_content = load_processed()
utility, user_ids, movie_ids, u2r, m2c = build_utility_matrix(ratings_cf)
n_users, n_movies = utility.shape
density = utility.nnz / (n_users * n_movies)
print(f"shape={utility.shape} nnz={utility.nnz:,} density={density:.4%}")
assert issparse(utility)
assert density < 0.01, "utility should stay sparse (<1%)"
pd.DataFrame([{
    "n_users": n_users,
    "n_movies": n_movies,
    "nnz": utility.nnz,
    "density": round(density, 6),
}])

## T17 — Item similarity sanity

Toy Story (1) vs Jumanji (2) should have positive cosine if both exist in CF vocab.

In [ ]:
cf = load_cf_artifacts()
print("item_sim shape:", cf.item_similarity.shape, "nnz:", cf.item_similarity.nnz)
for a, b, label in [(1, 2, "Toy vs Jumanji")]:
    if a in cf.movie_to_col and b in cf.movie_to_col:
        sim = float(cf.item_similarity[cf.movie_to_col[a], cf.movie_to_col[b]])
        print(f"{label}: {sim:.4f}")
    else:
        print(f"{label}: movie id missing from CF vocab")

# Smoke recommend for a known user
recs = recommend_for_user(cf, movies, user_id=1, top_k=5)
seen = set(ratings_cf.loc[ratings_cf.userId == 1, "movieId"])
assert set(recs.movieId).isdisjoint(seen)
recs[["movieId", "title", "score"]]

## T19 — Eval sweep (sample size)

Light sweep — full rebuild is expensive; use `run_evaluation` sample sizes.

In [ ]:
results = []
for sample in [50, 100, 200]:
    t0 = time.time()
    s = run_evaluation(ratings_cf, movies, sample_size=sample, top_k=10, min_rating=4.0)
    s = s.copy()
    s["sample"] = sample
    s["elapsed_s"] = round(time.time() - t0, 1)
    results.append(s)
    print(s.to_string(index=False))

out = pd.concat(results, ignore_index=True)
out_path = ROOT / "reports" / "cf_eval_scores.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(out_path, index=False)
print("saved", out_path)
out